In [124]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, RocCurveDisplay, classification_report
from sklearn.datasets import make_classification
from sklearn.svm import SVC
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

Goal: We want to maximize amount of money saved by Big G. A full derate costs around \\$4000 for towing, plus the cost of
repairs. The cost of a false positive prediction is about \\$500 due to having the truck oﬀ the road and serviced unnecessarily. Thus, we'll want to optimize our model for recall.

We want to attempt to predict an idle derate at least **2 hours** before it occurs.

In [125]:
faults = pd.read_csv("../data/J1939Faults.csv")

# Dropping columns that are not relevant
faults_clean = faults.drop(columns=['ecuSoftwareVersion', 'ecuSerialNumber', 'ecuModel', 'ecuMake', 'ecuSource', 'MCTNumber', 'ESS_Id', 'actionDescription', 'faultValue'])

# Concatenating spn and fmi into a new column
faults_clean['spn-fmi'] = faults_clean['spn'].astype(str) + "-" + faults_clean['fmi'].astype(str)

faults_clean.head()

/var/folders/51/zgq0lbb14t13h0_bgn8nntnm0000gn/T/ipykernel_64495/1086016911.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  faults = pd.read_csv("../data/J1939Faults.csv")


,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp,spn-fmi
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000,111-17
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000,629-12
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000,1807-2
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000,1807-2
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000,4364-17


Looking at the first record, here is a breakdown of the important values.

* ESS_Id, actionDescription, ecuSoftwareVersion, ecuSerialNumber, ecuModel, ecuMake, ecuSource, faultValue, and MCTNumber are unlikely to provide any predictive value.
* We can see the time of the event in the **EventTimeStamp** column. Note that this may be different from the **LocationTimeStamp** value, which indicates when the Latitude/Longitude values were recorded.
* The **spn** and **fmi** columns together indicate the type of fault, and there may be a description of that fault in the **eventDescription** column, although this column is sometimes missing.
* Faults are recorded when the light goes on and when it goes off, which is indicated by the **active** column, with True indicating the light turning on and False indicating turning off. The number of times the code has been set or unset is in the **faultValue** column, although this value can be unreliable. 
* Each truck has an identifier, the **EquipmentID** value.
* Each record can be linked to the on-board diagnostics data through the **RecordID** column.

Filtering out vehicles that are likely being serviced: (We'll come back to this)

Service stations are at (36.0666667, -86.4347222), (35.5883333, -86.4438888), and (36.1950, -83.174722)

In [126]:
# Creating a dataframe with service station locations
stations = pd.DataFrame({'lat':[36.0666667, 35.5883333, 36.1950], 'lon':[-86.4347222, -86.4438888, -83.174722]})

In [127]:
stations.loc[0]['lat']

np.float64(36.0666667)

In [128]:
# From this article: https://kanoki.org/2019/12/27/how-to-calculate-distance-in-python-and-pandas-using-scipy-spatial-and-distance-functions/
# Vectorized Haversine formula
def haversine_vectorize(lon1, lat1, lon2, lat2):

    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    newlon = lon2 - lon1
    newlat = lat2 - lat1

    haver_formula = np.sin(newlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(newlon/2.0)**2

    dist = 2 * np.arcsin(np.sqrt(haver_formula ))
    mi = 3958 * dist 
    return mi

faults_clean['dist_from_stat_1'] = haversine_vectorize(stations.loc[0]['lon'], stations.loc[0]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_2'] = haversine_vectorize(stations.loc[1]['lon'], stations.loc[1]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])
faults_clean['dist_from_stat_3'] = haversine_vectorize(stations.loc[2]['lon'], stations.loc[2]['lat'], faults_clean['Longitude'], faults_clean['Latitude'])


In [129]:
faults_clean.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,LocationTimeStamp,spn-fmi,dist_from_stat_1,dist_from_stat_2,dist_from_stat_3
0,1,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111,17,True,2,1439,38.857638,-84.626851,2015-02-21 11:34:25.000,111-17,216.779342,246.957471,200.394786
1,2,2015-02-21 11:34:34.000,NaN,629,12,True,127,1439,38.857638,-84.626851,2015-02-21 11:35:10.000,629-12,216.779342,246.957471,200.394786
2,3,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807,2,False,127,1369,41.421250,-87.767361,2015-02-21 11:35:26.000,1807-2,376.784933,409.225406,437.407343
3,4,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807,2,True,127,1369,41.421018,-87.767361,2015-02-21 11:36:08.000,1807-2,376.769223,409.209647,437.394356
4,5,2015-02-21 11:39:41.000,NaN,4364,17,False,2,1674,38.416481,-89.442638,2015-02-21 11:39:37.000,4364-17,231.732035,255.969764,376.935409


In [130]:
faults_clean.shape

(1187335, 15)

In [131]:
# Only keep rows that are more than 1 mile away from all 3 of the service stations
faults_clean = faults_clean[(faults_clean['dist_from_stat_1'] > 1) & (faults_clean['dist_from_stat_2'] > 1) & (faults_clean['dist_from_stat_3'] > 1)]


In [132]:
faults_clean.shape

(1055071, 15)

In [133]:
1187335-1055071

132264

In [134]:
(1187335-1055071)/1187335

0.11139568866410912

Looks like about 11% of our dataset was collected within 1 mile of a service station.

In [135]:
diagnostics = pd.read_csv("../data/VehicleDiagnosticOnboardData.csv")

diagnostics.head()

,Id,Name,Value,FaultId
0,1,IgnStatus,False,1
1,2,EngineOilPressure,0,1
2,3,EngineOilTemperature,96.74375,1
3,4,TurboBoostPressure,0,1
4,5,EngineLoad,11,1


Pivoting diagnostic onboard data by fault ID so that it's easier to merge. 

In [136]:
diagnostics[diagnostics['FaultId']==2]

,Id,Name,Value,FaultId
21,22,IgnStatus,True,2
22,23,LampStatus,1279,2


In [137]:
diagnostics_pivoted = diagnostics.pivot(index='FaultId', columns='Name', values='Value')

In [138]:
diagnostics_pivoted.head()

Name,AcceleratorPedal,BarometricPressure,CruiseControlActive,CruiseControlSetSpeed,DistanceLtd,EngineCoolantTemperature,EngineLoad,EngineOilPressure,EngineOilTemperature,EngineRpm,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
FaultId,,,,,,,,,,,,,,,,,,,,,
1,0,14.21,False,66.48672,423178.7,100.4,11,0,96.74375,0,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


To merge diagnostic info with fault code info, we'll match up the **RecordID** and **FaultId**.

In [139]:
faults_full = pd.merge(faults_clean, diagnostics_pivoted, left_on='RecordID', right_on='FaultId', how='outer')

faults_full.head()

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1.0,2015-02-21 10:47:13.000,Low (Severity Low) Engine Coolant Level,111.0,17.0,True,2.0,1439,38.857638,-84.626851,...,NaN,False,78.8,1023,True,NaN,0,3276.75,NaN,0
1,2.0,2015-02-21 11:34:34.000,NaN,629.0,12.0,True,127.0,1439,38.857638,-84.626851,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3.0,2015-02-21 11:35:31.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,False,127.0,1369,41.421250,-87.767361,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4.0,2015-02-21 11:35:33.000,Incorrect Data Steering Wheel Angle,1807.0,2.0,True,127.0,1369,41.421018,-87.767361,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,2015-02-21 11:39:41.000,NaN,4364.0,17.0,False,2.0,1674,38.416481,-89.442638,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


In [140]:
faults_full[faults_full['EquipmentID']==1674].sort_values(by='EventTimeStamp')

,RecordID,EventTimeStamp,eventDescription,spn,fmi,active,activeTransitionCount,EquipmentID,Latitude,Longitude,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
32805,33917.0,2015-04-25 07:00:05.000,Incorrect Data Axle Load Sensor,1059.0,2.0,False,127.0,1674,40.207592,-76.600879,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
32806,33918.0,2015-04-25 07:00:08.000,Incorrect Data Axle Load Sensor,1059.0,2.0,True,127.0,1674,40.207592,-76.600879,...,NaN,False,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
33019,34131.0,2015-04-25 12:20:55.000,Incorrect Data Axle Load Sensor,1059.0,2.0,False,127.0,1674,37.100740,-80.553009,...,NaN,NaN,NaN,255,NaN,NaN,NaN,NaN,NaN,NaN
33076,34188.0,2015-04-25 14:29:09.000,Incorrect Data Axle Load Sensor,1059.0,2.0,True,127.0,1674,36.779722,-81.750277,...,NaN,True,100.4,1279,NaN,NaN,65.59835,NaN,42.8,10.44
33135,34247.0,2015-04-25 17:11:51.000,Incorrect Data Axle Load Sensor,1059.0,2.0,False,127.0,1674,35.740648,-84.044722,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955458,984984.0,2018-03-17 21:03:58.000,Low Current Aftertreatment 1 Purge Air Actuator,3490.0,5.0,False,1.0,1674,33.619351,-84.328148,...,NaN,NaN,NaN,255,NaN,NaN,NaN,NaN,NaN,NaN
963254,993762.0,2018-04-03 16:33:22.000,Low (Severity High) Engine Coolant Level,111.0,1.0,True,56.0,1674,37.079675,-80.678055,...,NaN,True,113,4351,NaN,NaN,16.20662,NaN,62,15.37
963266,993774.0,2018-04-03 16:56:55.000,Low (Severity High) Engine Coolant Level,111.0,1.0,False,60.0,1674,37.086157,-80.645185,...,NaN,NaN,NaN,4351,NaN,NaN,NaN,NaN,NaN,NaN
965681,996189.0,2018-04-10 00:29:02.000,Low Current Aftertreatment 1 Purge Air Actuator,3490.0,5.0,True,2.0,1674,36.199537,-83.026527,...,NaN,True,127.4,255,NaN,NaN,9.218624,NaN,28.8,5.22


Creating target column:

Our plan will be to make 4 different target columns: 
* Derate in 0-2 hours
* Derate in 2-6 hours
* Derate in 6-12 hours
* Derate in 12-24 hours

Then, we'll fit our model on each of the 4 target columns and see which model is most predictive.

The idea is to give the vehicle operator enough warning so that they have enough time to drive to a service station. 0-2 hours of warning probably isn't enough time. As we increase the window larger and larger, we'll probably lose some predictive power. So we'll try to find the sweet spot by evaluating several different time windows.

In [141]:
# Creating a new column to mark whether the fault represents a full derate
# We'll use this when we apply .rolling later 
faults_full['derate_flag'] = ((faults_full["spn"] == 5246) & (faults_full["active"] == True)).astype('int')

In [142]:
# Converting EventTimeStamp to datetime 
faults_full['EventTimeStamp']= pd.to_datetime(faults_full['EventTimeStamp'])

In [143]:
# Running into an issue with duplicated EquipmentID/EventTimeStamp. 
# The .rolling method can't handle the duplicated EquipmentID and EventTimeStamp columns.
# We'll need to de-duplicate before we apply .rolling
# By taking the max of 'derate_flag', we're ensuring that for each EquipmentID/EventTimeStamp combination, 
#       we're capturing whether at least one derate occurred for that vehicle at that time (if multiple faults occurred simultaneously)
faults_dedup = faults_full.groupby(['EquipmentID', 'EventTimeStamp']).max('derate_flag').reset_index()

In [144]:
faults_dedup.head()

,EquipmentID,EventTimeStamp,RecordID,spn,fmi,activeTransitionCount,Latitude,Longitude,dist_from_stat_1,dist_from_stat_2,dist_from_stat_3,derate_flag
0,301,2015-05-11 13:11:20,49415.0,639.0,2.0,127.0,36.189398,-82.795601,203.213841,208.343967,21.139706,0
1,301,2015-05-13 08:22:32,51363.0,596.0,31.0,3.0,35.872500,-84.475648,110.345267,112.103834,76.011053,0
2,301,2015-05-18 09:34:05,57330.0,3226.0,10.0,6.0,35.972870,-83.920555,140.619456,143.881678,44.375304,0
3,301,2015-05-21 13:57:35,61706.0,639.0,2.0,127.0,36.384953,-86.478379,22.121444,55.064352,184.408522,0
4,301,2015-05-21 14:54:32,61801.0,639.0,2.0,127.0,36.384814,-86.478379,22.111900,55.054756,184.408003,0


In [168]:
# By sorting the EventTimeStamp in descending order, we're ensuring that we are rolling forwards in time (since .rolling looks at preceding rows)
# Then we're applying the .rolling method with a certain time window and taking the rolling count of derates.
# The new columns are indicating the total number of derates that happened in the next certain time window for the given vehicle.
target = faults_dedup[['EquipmentID','EventTimeStamp']].copy()
                       
target['total_derates_2_hr'] = faults_dedup.sort_values(by='EventTimeStamp', ascending=False).groupby('EquipmentID').rolling(window='2h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_6_hr'] = faults_dedup.sort_values(by='EventTimeStamp', ascending=False).groupby('EquipmentID').rolling(window='6h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_12_hr'] = faults_dedup.sort_values(by='EventTimeStamp', ascending=False).groupby('EquipmentID').rolling(window='12h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)
target['total_derates_24_hr'] = faults_dedup.sort_values(by='EventTimeStamp', ascending=False).groupby('EquipmentID').rolling(window='24h', on='EventTimeStamp')['derate_flag'].sum().reset_index(drop=True)

# Next we take successive differences so that we can get total number of derates within the time ranges we're interested in (2-6 hrs, 6-12 hrs, 12-24 hrs)
# If there were 1 or more derates in the given time range, we mark it with "1"; otherwise, "0"
target['derate_2_6_hr'] = np.where(faults_dedup['total_derates_6_hr'] - faults_dedup['total_derates_2_hr'] > 0, 1, 0)
target['derate_6_12_hr'] = np.where(faults_dedup['total_derates_12_hr'] - faults_dedup['total_derates_6_hr'] > 0, 1, 0)
target['derate_12_24_hr'] = np.where(faults_dedup['total_derates_24_hr'] - faults_dedup['total_derates_12_hr'] > 0, 1, 0)

In [169]:
# Drop intermediate columns that we don't need
target = target.drop(columns=['total_derates_2_hr','total_derates_6_hr','total_derates_12_hr','total_derates_24_hr'])

In [170]:
# Next, we'll merge the new target columns into the original DataFrame, merging on EventTimeStamp and EquipmentID
# Any duplicate EventTimeStamp/EquipmentID combinations will have the same value in the target column
faults_full = pd.merge(target, faults_full, on=['EventTimeStamp','EquipmentID'], how='right')

In [171]:
faults_full.head()

,EquipmentID,EventTimeStamp,derate_2_6_hr,derate_6_12_hr,derate_12_24_hr,derates_2_6_hr,derates_6_12_hr,derates_12_24_hr,RecordID,eventDescription,...,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure,derate_flag
0,1439,2015-02-21 10:47:13,0.0,0.0,0.0,0.0,0.0,0.0,1.0,Low (Severity Low) Engine Coolant Level,...,False,78.8,1023,True,NaN,0,3276.75,NaN,0,0
1,1439,2015-02-21 11:34:34,0.0,0.0,0.0,0.0,0.0,0.0,2.0,NaN,...,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
2,1369,2015-02-21 11:35:31,0.0,0.0,0.0,0.0,0.0,0.0,3.0,Incorrect Data Steering Wheel Angle,...,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
3,1369,2015-02-21 11:35:33,0.0,0.0,0.0,0.0,0.0,0.0,4.0,Incorrect Data Steering Wheel Angle,...,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1674,2015-02-21 11:39:41,0.0,0.0,0.0,0.0,0.0,0.0,5.0,NaN,...,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN,0


In [89]:
faults_full.columns

Index(['RecordID', 'EventTimeStamp', 'eventDescription', 'spn', 'fmi',
       'active', 'activeTransitionCount', 'EquipmentID', 'Latitude',
       'Longitude', 'LocationTimeStamp', 'spn-fmi', 'dist_from_stat_1',
       'dist_from_stat_2', 'dist_from_stat_3', 'AcceleratorPedal',
       'BarometricPressure', 'CruiseControlActive', 'CruiseControlSetSpeed',
       'DistanceLtd', 'EngineCoolantTemperature', 'EngineLoad',
       'EngineOilPressure', 'EngineOilTemperature', 'EngineRpm',
       'EngineTimeLtd', 'FuelLevel', 'FuelLtd', 'FuelRate', 'FuelTemperature',
       'IgnStatus', 'IntakeManifoldTemperature', 'LampStatus', 'ParkingBrake',
       'Speed', 'SwitchedBatteryVoltage', 'Throttle', 'TurboBoostPressure',
       'derate_flag'],
      dtype='object')

Exploring missing data:

In [58]:
# Looking at nan proportions overall 
faults_full.isnull().mean().sort_values(ascending=False)

ServiceDistance              0.999819
SwitchedBatteryVoltage       0.903937
FuelTemperature              0.748083
ParkingBrake                 0.663135
Throttle                     0.645843
FuelLevel                    0.576535
AcceleratorPedal             0.552031
CruiseControlActive          0.515793
CruiseControlSetSpeed        0.514494
EngineTimeLtd                0.510361
TurboBoostPressure           0.508689
EngineOilTemperature         0.508216
Speed                        0.508213
FuelLtd                      0.507136
FuelRate                     0.507100
EngineLoad                   0.506777
DistanceLtd                  0.506610
BarometricPressure           0.506478
EngineCoolantTemperature     0.506398
EngineOilPressure            0.506252
IntakeManifoldTemperature    0.506213
EngineRpm                    0.505682
IgnStatus                    0.487546
eventDescription             0.154188
EquipmentID                  0.111396
EventTimeStamp               0.111396
dist_from_st

In [86]:
# Dropping service distance since it is mostly nulls
faults_full = faults_full.drop(columns=['ServiceDistance'])

Don't do K nearest neighbors with the ltd columns (try forward fill).

Time series K-fold method would be the most conservative when it comes to evaluating different models.

In [92]:
faults_full.dtypes

RecordID                            float64
EventTimeStamp               datetime64[ns]
eventDescription                     object
spn                                 float64
fmi                                 float64
active                               object
activeTransitionCount               float64
EquipmentID                          object
Latitude                            float64
Longitude                           float64
LocationTimeStamp                    object
spn-fmi                              object
dist_from_stat_1                    float64
dist_from_stat_2                    float64
dist_from_stat_3                    float64
AcceleratorPedal                     object
BarometricPressure                   object
CruiseControlActive                  object
CruiseControlSetSpeed                object
DistanceLtd                          object
EngineCoolantTemperature             object
EngineLoad                           object
EngineOilPressure               

In [101]:
faults_full['EngineCoolantTemperature']

0          100.4
1            NaN
2            NaN
3            NaN
4            NaN
           ...  
1187330      NaN
1187331      185
1187332    186.8
1187333    181.4
1187334      NaN
Name: EngineCoolantTemperature, Length: 1187335, dtype: object

In [99]:
faults_full[['TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','DistanceLtd','EngineCoolantTemperature']] = faults_full[['TurboBoostPressure','AcceleratorPedal','BarometricPressure','CruiseControlSetSpeed','DistanceLtd','EngineCoolantTemperature']].astype('float64')

ValueError: could not convert string to float: '33,35'

In [ ]:
# We'll apply a StandardScaler to numeric features
numeric_features = []
numeric_transformer = Pipeline(
    steps=[("scaler", StandardScaler())]
)

# We'll apply a one-hot encoder to the categorical features
categorical_features = []
categorical_transformer = Pipeline(
    steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Append KNN to preprocessing pipeline to fill in missing values 
# Need to figure out how to deal with nulls in the ltd columns
dataprep = Pipeline(
    steps=[("preprocessor", preprocessor), ("knn", KNNImputer(n_neighbors=3))]
)

We could possibly add a missing value indicator while replacing null values, to see if it helps our model.

We can get a little bit more information about the different fault codes from the Service Fault Codes spreadsheet. We'll come back to this information later to see if we think it would be a helpful addition to our model. The problem is there can be multiple rows for a given fault code, so it's unclear which information should be used. As a stretch goal, we could analyze the algorithm description or lamp color to bucket issues into mild/moderate/severe categories.

In [27]:
sfc = pd.read_excel("../data/Service Fault Codes_1_0_0_167.xlsx")
sfc.head()

/home/michael/anaconda3/lib/python3.9/site-packages/openpyxl/worksheet/_read_only.py:79: UserWarning: Data Validation extension is not supported and will be removed
  for idx, row in parser.parse():


,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
1,Y,112,167,Not Mapped,20,128,7,635,7,Not Mapped,Red,Stop / Shutdown,Engine Timing Actuator Driver Circuit - Mechan...,Mechanical failure in the engine timing actuat...
2,Y,113,167,Not Mapped,20,128,3,635,3,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,High signal voltage detected at the engine tim...
3,Y,114,167,Not Mapped,20,128,4,635,4,Not Mapped,Amber,Warning,Engine Timing Actuator Driver Circuit - Voltag...,Low voltage detected at the engine timing actu...
4,Y,115,167,190,Not Mapped,Not Mapped,2,612,2,P0008,Red,Stop / Shutdown,Engine Magnetic Speed/Position Lost Both of Tw...,The ECM has detected that the primary and back...


For a large number of fault codes, there are multiple records. For example, if we look at the rows for the first fault in our dataset, we see that there are two rows.

In [28]:
(
    sfc
    .loc[sfc['SPN'] == 111]
    .loc[sfc['J1939 FMI'] == 17]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
1589,Y,2448,167,111,Not Mapped,Not Mapped,1,111,17,P2560,Maintenance,Maintenance,Coolant Level - Data Valid But Below Normal Op...,Low engine coolant level detected.
3540,Y,5167,167,Not Mapped,Not Mapped,Not Mapped,1,111,17,Not Mapped,Amber,Warning,Coolant Level - Data Valid But Below Normal Op...,NaN


Or even more.

In [29]:
(
    sfc
    .loc[sfc['SPN'] == 629]
    .loc[sfc['J1939 FMI'] == 12]
)

,Published in CES 14602,Cummins Fault Code,Revision,PID,SID,MID,J1587 FMI,SPN,J1939 FMI,J2012 Pcode,Lamp Color,Lamp Device,Cummins Description,Algorithm Description
0,Y,111,167,Not Mapped,254,0,12,629,12,P0606,Red,Stop / Shutdown,Engine Control Module Critical Internal Failur...,Error internal to the ECM related to memory ha...
180,Y,343,167,Not Mapped,254,0,12,629,12,P0607,Amber,Warning,Engine Control Module Warning Internal Hardwar...,ECM power supply errors have been detected.
689,Y,1116,167,Not Mapped,254,0,12,629,12,Not Mapped,Amber,Warning,Engine Control Module Critical Internal Failur...,ECM Internal failure has occurred.
854,Y,1388,167,Not Mapped,254,0,12,629,12,Not Mapped,NaN,NaN,Engine Control Module Data Lost - Bad Intellig...,The ECM data has been lost.
1019,Y,1597,167,Not Mapped,254,0,12,629,12,Not Mapped,Maintenance,Maintenance,Engine Control Module Critical Internal Failur...,The ECM has occurred an internal failure.
